In order to interact with Google Earth Engine we will need to install the relevant dependencies.  We will do this by building a Conda environment from the included `environment.yml`.  

<details>
  <summary>What is a Conda environment?</summary>
  <p>A Conda environment is an isolated workspace that has its own version of Python and its own installed packages. It lets you set up the exact tools needed for a project without interfering with other projects or the system Python. For more detail see <a href="https://www.anaconda.com/docs/getting-started/concepts/what-is-an-environment">this Anaconda explanation</a>.</p>
</details>

to build the environment, open Anaconda Powershell and type, 

```
cd C:\path\to\NR4418\gee_example
conda env create -f environment.yml
conda activate ee
python -m ipykernel install --user --name ee --display-name "Python (ee)"
```

Then we will authenticate the Earth Engine connection,

```
earthengine authenticate
```
After you run `earthengine authenticate`, a browser should open. You must sign into the Google account you will use for Earth Engine, grant the requested permissions, then return to PowerShell; it should report that the authorization token was saved.  If the browser does not open, copy the URL shown in PowerShell into a browser.


After installing dependencies we can then import the modules needed for Earth Engine, geemap, and vector AOI handling.

In [21]:
# standard modules
import json
from pathlib import Path
import time

# specialized modules
import ee
import geemap
import geopandas as gpd
import pandas as pd
from shapely.geometry import box
from tqdm import tqdm

# initialize the Earth Engine module.
ee.Initialize(project='remote-sensing-484922')


# read Camp Fire perimeter
perimeter_path = Path().cwd() / 'camp_fire_2018_epsg4326.geojson'


We will read the geojson of the perimeter path and find the extent

In [22]:
perimeter = gpd.read_file(perimeter_path)

# make sure CRS is 4326
if perimeter.crs is None:
    perimeter = perimeter.set_crs(4326)
else:
    perimeter = perimeter.to_crs(4326)

# Use the extent of the perimeter as the AOI.
minx, miny, maxx, maxy = perimeter.total_bounds
aoi = gpd.GeoDataFrame(
    {'name': ['camp_fire_2018_extent']},
    geometry=[box(minx, miny, maxx, maxy)],
    crs='EPSG:4326',
)

# load json into dict
perimeter_json = json.loads(perimeter[['geometry']].to_json())
aoi_json = json.loads(aoi[['geometry']].to_json())

# change from dict to ee featurecollection
gee_perimeter = geemap.geojson_to_ee(perimeter_json)
gee_aoi = geemap.geojson_to_ee(aoi_json)

# display bounds
aoi.total_bounds


array([-121.77782801,   39.59858538, -121.35260516,   39.89781567])

Lts take a look the perimeter and bounds.

In [23]:
# inspect the GeoJSON and derived extent as EEObjects through geemap.
test_map = geemap.Map()
test_map.centerObject(gee_aoi, 10)
test_map.addLayer(
    gee_aoi.style(color='22C55E', fillColor='00000000', width=2),
    {},
    'AOI extent',
)
test_map.addLayer(
    gee_perimeter.style(color='ff0000', fillColor='00000000', width=2),
    {},
    'Camp Fire perimeter',
)

test_map


Map(center=[39.74815687150269, -121.56521658485447], controls=(WidgetControl(options=['position', 'transparent…

In [24]:
# get extent
verts = [
    [minx, miny],
    [minx, maxy],
    [maxx, maxy],
    [maxx, miny],
    [minx, miny],
]

# make normal floats from numpy float64 values
verts = [[float(x), float(y)] for x, y in verts]
gee_extent = ee.Geometry.Polygon([verts], proj='EPSG:4326', geodesic=False)

verts


[[-121.777828008347, 39.5985853827284],
 [-121.777828008347, 39.8978156744478],
 [-121.352605161362, 39.8978156744478],
 [-121.352605161362, 39.5985853827284],
 [-121.777828008347, 39.5985853827284]]

Next we will search the catalogue for NAIP imagery for the AOI for a little before and after Novemebr 2018.

In [25]:
# date range
# The Camp Fire started in November 2018, so this range searches the fire year
# and later NAIP collections likely to include post-fire imagery.
START = ee.Date('2017-01-01')
END = ee.Date('2020-12-31')

# date and geographic filter
col_filter = ee.Filter.And(
    ee.Filter.geometry(gee_extent),
    ee.Filter.date(START, END),
)

naip = (
    ee.ImageCollection('USDA/NAIP/DOQQ')
    .filter(col_filter)
    .select(['R', 'G', 'B', 'N'])
)

naip_2018 = naip.filterDate('2018-01-01', '2019-01-01')
naip_2020 = naip.filterDate('2020-01-01', '2021-01-01')

# Display the available assets before mosaicking, mapping, or exporting.
naip_asset_indexes = naip.aggregate_array('system:index').getInfo()
naip_dates = (
    naip.aggregate_array('system:time_start')
    .map(lambda t: ee.Date(t).format('YYYY-MM-dd'))
    .getInfo()
)

available_assets = pd.DataFrame({
    'date': naip_dates,
    'asset_id': [f'USDA/NAIP/DOQQ/{asset}' for asset in naip_asset_indexes],
    'system_index': naip_asset_indexes,
}).sort_values(['date', 'asset_id']).reset_index(drop=True)

print(f'NAIP images found: {len(available_assets)}')
available_assets


NAIP images found: 96


,date,asset_id,system_index
0,2018-07-16,USDA/NAIP/DOQQ/m_3912102_se_10_060_20180716_20...,m_3912102_se_10_060_20180716_20190210
1,2018-07-16,USDA/NAIP/DOQQ/m_3912110_ne_10_060_20180716_20...,m_3912110_ne_10_060_20180716_20190210
2,2018-07-16,USDA/NAIP/DOQQ/m_3912110_se_10_060_20180716_20...,m_3912110_se_10_060_20180716_20190210
3,2018-07-16,USDA/NAIP/DOQQ/m_3912118_ne_10_060_20180716_20...,m_3912118_ne_10_060_20180716_20190209
4,2018-07-16,USDA/NAIP/DOQQ/m_3912118_se_10_060_20180716_20...,m_3912118_se_10_060_20180716_20190209
...,...,...,...
91,2020-07-10,USDA/NAIP/DOQQ/m_3912120_ne_10_060_20200710,m_3912120_ne_10_060_20200710
92,2020-07-10,USDA/NAIP/DOQQ/m_3912120_se_10_060_20200710,m_3912120_se_10_060_20200710
93,2020-07-10,USDA/NAIP/DOQQ/m_3912121_nw_10_060_20200710,m_3912121_nw_10_060_20200710
94,2020-07-10,USDA/NAIP/DOQQ/m_3912121_sw_10_060_20200710,m_3912121_sw_10_060_20200710


In [26]:
def image_to_tile_feature(image):
    image = ee.Image(image)
    return ee.Feature(
        image.geometry(),
        {
            'system_index': image.get('system:index'),
            'date': ee.Date(image.get('system:time_start')).format('YYYY-MM-dd'),
        },
    )


def collection_to_tile_features(collection):
    return ee.FeatureCollection(
        collection.toList(collection.size()).map(image_to_tile_feature)
    )


naip_tiles_2018 = collection_to_tile_features(naip_2018)
naip_tiles_2020 = collection_to_tile_features(naip_2020)

tile_rgb_vis = {
    'bands': ['R', 'G', 'B'],
    'min': 0,
    'max': 255,
}

tile_map = geemap.Map()
tile_map.centerObject(gee_aoi, 10)
tile_map.addLayer(naip_2018, tile_rgb_vis, 'NAIP 2018 image tiles')
tile_map.addLayer(naip_2020, tile_rgb_vis, 'NAIP 2020 image tiles', False)
tile_map.addLayer(
    naip_tiles_2018.style(color='00FF00', fillColor='00000000', width=2),
    {},
    'NAIP 2018 tile footprints',
)
tile_map.addLayer(
    naip_tiles_2020.style(color='00BFFF', fillColor='00000000', width=2),
    {},
    'NAIP 2020 tile footprints',
    False,
)
tile_map.addLayer(
    gee_perimeter.style(color='ff0000', fillColor='00000000', width=2),
    {},
    'Camp Fire perimeter',
)
tile_map.addLayer(
    gee_aoi.style(color='22C55E', fillColor='00000000', width=2),
    {},
    'AOI extent',
)

tile_map


Map(center=[39.74815687150269, -121.56521658485447], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
# NAIP is 1 m imagery, so a full extent export can be large.
# Increase scale to 2 or 5 for a smaller, faster export.
# Sort by date so newer imagery is drawn on top when each year is mosaicked.
naip_mosaic_2018 = naip_2018.sort('system:time_start').mosaic().clip(gee_aoi)
naip_mosaic_2020 = naip_2020.sort('system:time_start').mosaic().clip(gee_aoi)

export_params_2018 = {
    'image': naip_mosaic_2018,
    'description': 'camp_fire_naip_2018_mosaic',
    'folder': 'nr218',
    'fileNamePrefix': 'camp_fire_naip_2018_mosaic',
    'scale': 1,
    'region': gee_aoi.geometry(),
    'fileFormat': 'GeoTIFF',
    'maxPixels': 1e13,
}

export_params_2020 = {
    'image': naip_mosaic_2020,
    'description': 'camp_fire_naip_2020_mosaic',
    'folder': 'nr218',
    'fileNamePrefix': 'camp_fire_naip_2020_mosaic',
    'scale': 1,
    'region': gee_aoi.geometry(),
    'fileFormat': 'GeoTIFF',
    'maxPixels': 1e13,
}

task_2018 = ee.batch.Export.image.toDrive(**export_params_2018)
task_2020 = ee.batch.Export.image.toDrive(**export_params_2020)

task_2018.start()
task_2020.start()

tasks = [task_2018, task_2020]


In [ ]:
for task in tasks:
    info = task.status()
    print(f"{info['description']}: {info['state']}", info.get('progress', ''))
